# EfficientNet B0 — PM2.5 Classification with Train-Test Split Sensitivity Analysis
Runs 9 different train/test splits (90:10 → 10:90) to produce a learning curve.

In [ ]:
# ─────────────────────────────────────────────
# Cell 1 — Imports
# ─────────────────────────────────────────────
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve, auc
)

print('All imports successful.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 2 — Configuration
# ─────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

CONFIG = {
    'model_name'    : 'EfficientNetB0',
    'batch_size'    : 32,
    'learning_rate' : 0.001,
    'epochs'        : 300,
    'image_size'    : 224,
    'random_state'  : 42,
}

# 9 splits: 90:10 → 10:90
SPLIT_RATIOS = [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]

# Dataset paths
TRAIN_CSV  = '/kaggle/input/datasets/deadcardassian/pm25vision/train/metadata.csv'
TRAIN_IMGS = '/kaggle/input/datasets/deadcardassian/pm25vision/train/images'
TEST_CSV   = '/kaggle/input/datasets/deadcardassian/pm25vision/test/metadata.csv'
TEST_IMGS  = '/kaggle/input/datasets/deadcardassian/pm25vision/test/images'

for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 — Dataset Class
#   Accepts a DataFrame directly (not a CSV path)
#   so we can pass split subsets straight in.
# ─────────────────────────────────────────────
class PM25Dataset(Dataset):
    def __init__(self, dataframe, transform=None, class_to_idx=None):
        """
        Args:
            dataframe    : pandas DataFrame with columns [filename, image_dir, pm25_bin]
            transform    : torchvision transforms
            class_to_idx : pass the training set mapping to the test set so
                           both use identical label indices
        """
        self.df = dataframe.copy().reset_index(drop=True)
        self.transform = transform

        # Remove rows whose image file does not exist
        self.df = self.df[
            self.df.apply(
                lambda r: os.path.exists(os.path.join(r['image_dir'], r['filename'])),
                axis=1
            )
        ].reset_index(drop=True)

        # Build or reuse class → index mapping
        if class_to_idx is None:
            self.classes = sorted(
                self.df['pm25_bin'].unique(),
                key=lambda x: int(x.split('\u2013')[0])   # sort by lower bound
            )
            self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        else:
            self.class_to_idx = class_to_idx
            self.classes = list(class_to_idx.keys())

        self.df['label'] = self.df['pm25_bin'].map(self.class_to_idx)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(row['image_dir'], row['filename'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = int(row['label'])
        return image, label

print('PM25Dataset class defined.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 4 — Build Full Dataset DataFrame
#   Merge original train + test CSVs into one
#   pool, then we re-split it 9 different ways.
# ─────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

train_df['image_dir'] = TRAIN_IMGS
test_df['image_dir']  = TEST_IMGS

full_df = pd.concat([train_df, test_df], ignore_index=True)

print(f'Total samples in full pool : {len(full_df)}')
print(f'Class distribution:')
print(full_df['pm25_bin'].value_counts().sort_index())

In [ ]:
# ─────────────────────────────────────────────
# Cell 5 — Transforms
# ─────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

print('Transforms defined.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 6 — Helper: build a fresh model
# ─────────────────────────────────────────────
def build_model(num_classes):
    """Returns a fresh EfficientNet-B0 with frozen backbone
    and a new classification head for `num_classes` classes."""
    m = models.efficientnet_b0(weights='IMAGENET1K_V1')
    for param in m.parameters():
        param.requires_grad = False          # freeze backbone
    in_features = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(in_features, num_classes)   # new head
    return m.to(device)

print('build_model() helper defined.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 7 — Main Loop: 9 Splits
# ─────────────────────────────────────────────
all_results = []   # stores one dict per split

for train_ratio in SPLIT_RATIOS:
    test_ratio  = round(1 - train_ratio, 1)
    split_label = f"{int(train_ratio*100)}:{int(test_ratio*100)}"

    print(f"\n{'='*65}")
    print(f"  Split  {split_label}  |  "
          f"Train {int(train_ratio*100)}%  /  Test {int(test_ratio*100)}%")
    print(f"{'='*65}")

    # ── 1. Split ────────────────────────────────────────────────
    train_df_split, test_df_split = train_test_split(
        full_df,
        train_size   = train_ratio,
        random_state = CONFIG['random_state'],
        stratify     = full_df['pm25_bin']    # keeps class balance
    )
    print(f"  Train samples : {len(train_df_split)} | "
          f"Test samples : {len(test_df_split)}")

    # ── 2. Datasets & DataLoaders ────────────────────────────────
    train_dataset = PM25Dataset(train_df_split, transform=transform)
    test_dataset  = PM25Dataset(
        test_df_split,
        transform    = transform,
        class_to_idx = train_dataset.class_to_idx   # reuse mapping
    )

    num_classes = len(train_dataset.classes)
    class_names = train_dataset.classes

    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG['batch_size'],
        shuffle=True,  num_workers=2
    )
    test_loader = DataLoader(
        test_dataset, batch_size=CONFIG['batch_size'],
        shuffle=False, num_workers=2
    )

    # ── 3. Model, Loss, Optimiser ────────────────────────────────
    model     = build_model(num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(), lr=CONFIG['learning_rate']
    )

    # ── 4. Training ──────────────────────────────────────────────
    train_loss_hist, val_loss_hist   = [], []
    train_acc_hist,  val_acc_hist    = [], []

    start_train = time.time()

    for epoch in range(CONFIG['epochs']):
        # — train phase —
        model.train()
        running_loss = 0.0
        train_correct = train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss  += loss.item()
            _, preds       = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

        avg_train_loss = running_loss / len(train_loader)
        train_acc      = train_correct / train_total
        train_loss_hist.append(avg_train_loss)
        train_acc_hist.append(train_acc)

        # — validation phase (using test split) —
        model.eval()
        val_loss = 0.0
        val_correct = val_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs  = model(images)
                loss     = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds  = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)

        avg_val_loss = val_loss / len(test_loader)
        val_acc      = val_correct / val_total
        val_loss_hist.append(avg_val_loss)
        val_acc_hist.append(val_acc)

        if (epoch + 1) % 50 == 0:
            print(f"  Epoch [{epoch+1:3d}/{CONFIG['epochs']}]  "
                  f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.4f}  |  "
                  f"Val Loss: {avg_val_loss:.4f}  Acc: {val_acc:.4f}")

    train_time = time.time() - start_train

    # ── 5. Final Evaluation ──────────────────────────────────────
    model.eval()
    y_true, y_pred, y_probs = [], [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1)
            preds   = torch.argmax(outputs, dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    y_true  = np.array(y_true)
    y_pred  = np.array(y_pred)
    y_probs = np.array(y_probs)

    acc       = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall    = recall_score(y_true, y_pred,    average='macro', zero_division=0)
    f1        = f1_score(y_true, y_pred,        average='macro', zero_division=0)
    cm        = confusion_matrix(y_true, y_pred)
    auc_score = roc_auc_score(y_true, y_probs,  multi_class='ovr')

    print(f"\n  ── Results (Split {split_label}) ──")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  AUC       : {auc_score:.4f}")
    print(f"  Train Time: {train_time:.1f}s")

    all_results.append({
        'split'         : split_label,
        'train_pct'     : int(train_ratio * 100),
        'test_pct'      : int(test_ratio  * 100),
        'train_samples' : len(train_df_split),
        'test_samples'  : len(test_df_split),
        'accuracy'      : acc,
        'precision'     : precision,
        'recall'        : recall,
        'f1'            : f1,
        'auc'           : auc_score,
        'train_time_s'  : round(train_time, 2),
        'train_loss_hist': train_loss_hist,
        'val_loss_hist'  : val_loss_hist,
        'train_acc_hist' : train_acc_hist,
        'val_acc_hist'   : val_acc_hist,
        'confusion_matrix': cm.tolist(),
        'class_names'    : class_names,
    })

print(f"\n{'='*65}")
print('All 9 splits completed!')
print(f"{'='*65}")

In [ ]:
# ─────────────────────────────────────────────
# Cell 8 — Summary Table
# ─────────────────────────────────────────────
summary_df = pd.DataFrame([{
    'Split'          : r['split'],
    'Train Samples'  : r['train_samples'],
    'Test Samples'   : r['test_samples'],
    'Accuracy'       : round(r['accuracy'],  4),
    'Precision'      : round(r['precision'], 4),
    'Recall'         : round(r['recall'],    4),
    'F1-Score'       : round(r['f1'],        4),
    'AUC'            : round(r['auc'],       4),
    'Train Time (s)' : r['train_time_s'],
} for r in all_results])

print('\nLearning Curve Summary — EfficientNet B0')
print('=' * 90)
print(summary_df.to_string(index=False))
print('=' * 90)

In [ ]:
# ─────────────────────────────────────────────
# Cell 9 — Learning Curve Plot
#   Accuracy / F1 / AUC vs Training %
# ─────────────────────────────────────────────
train_pcts = [r['train_pct'] for r in all_results]
accs       = [r['accuracy']  for r in all_results]
f1s        = [r['f1']        for r in all_results]
aucs       = [r['auc']       for r in all_results]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(train_pcts, accs,  marker='o', linewidth=2, label='Accuracy')
ax.plot(train_pcts, f1s,   marker='s', linewidth=2, label='F1-Score (macro)')
ax.plot(train_pcts, aucs,  marker='^', linewidth=2, label='AUC (OvR)')

ax.set_xlabel('Training Data (%)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('EfficientNet B0 — Learning Curve (9 Splits)', fontsize=14, fontweight='bold')
ax.set_xticks(train_pcts)
ax.set_xticklabels([f'{p}%' for p in train_pcts])
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('efficientnetb0_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: efficientnetb0_learning_curve.png')

In [ ]:
# ─────────────────────────────────────────────
# Cell 10 — Training Curves per Split
#   Loss & Accuracy over epochs for each split
# ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, r in enumerate(all_results):
    ax  = axes[i]
    eps = range(1, len(r['train_loss_hist']) + 1)

    ax2 = ax.twinx()
    ax.plot(eps,  r['train_loss_hist'], color='steelblue',  linewidth=1.5, label='Train Loss')
    ax.plot(eps,  r['val_loss_hist'],   color='coral',      linewidth=1.5, label='Val Loss', linestyle='--')
    ax2.plot(eps, r['train_acc_hist'],  color='green',      linewidth=1.5, label='Train Acc', linestyle='-.')
    ax2.plot(eps, r['val_acc_hist'],    color='purple',     linewidth=1.5, label='Val Acc',   linestyle=':')

    ax.set_title(f"Split {r['split']}", fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss',     color='steelblue')
    ax2.set_ylabel('Accuracy', color='green')
    ax2.set_ylim(0, 1.05)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='upper right')
    ax.grid(alpha=0.2, linestyle='--')

plt.suptitle('EfficientNet B0 — Training Curves per Split', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('efficientnetb0_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: efficientnetb0_training_curves.png')

In [ ]:
# ─────────────────────────────────────────────
# Cell 11 — Confusion Matrices (3 key splits)
#   90:10  /  50:50  /  10:90
# ─────────────────────────────────────────────
key_splits = [0, 4, 8]   # indices → 90:10, 50:50, 10:90
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, idx in zip(axes, key_splits):
    r  = all_results[idx]
    cm = np.array(r['confusion_matrix'])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    im = ax.imshow(cm_norm, cmap='Blues')
    ax.set_title(f"Split {r['split']} — Confusion Matrix", fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks(range(len(r['class_names'])))
    ax.set_yticks(range(len(r['class_names'])))
    ax.set_xticklabels(r['class_names'], rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(r['class_names'], fontsize=8)
    plt.colorbar(im, ax=ax)

plt.suptitle('Confusion Matrices at Key Splits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('efficientnetb0_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: efficientnetb0_confusion_matrices.png')

In [ ]:
# ─────────────────────────────────────────────
# Cell 12 — Save All Results to JSON & CSV
# ─────────────────────────────────────────────
# JSON (includes full history per split)
with open('efficientnetb0_all_results.json', 'w') as f:
    json.dump(all_results, f, indent=4)
print('Saved: efficientnetb0_all_results.json')

# CSV (summary metrics only)
summary_df.to_csv('efficientnetb0_summary.csv', index=False)
print('Saved: efficientnetb0_summary.csv')

print('\nDone! All outputs saved.')